
# Core Notebook 2 · Calendars, Day-Counts, and Schedules

Dates are treated as first-class data: calendars, business-day conventions, day-count contexts, and schedule builders guarantee deterministic accrual logic across instruments.


## Imports

In [1]:

from datetime import date

from finstack.core import dates

us_calendar = dates.get_calendar("usny")
print("Loaded calendar:", us_calendar.code, us_calendar.name)


Loaded calendar: usny United States (New York Federal) Holidays


## Calendar basics

In [2]:

codes = dates.available_calendar_codes()
print("Calendar sample:", codes[:8])

july4 = date(2025, 7, 4)
print("Is 4 July 2025 a business day?", us_calendar.is_business_day(july4))
adjusted = dates.adjust(july4, dates.BusinessDayConvention.PRECEDING, us_calendar)
print("Adjusted with Preceding:", adjusted)


Calendar sample: ['asx', 'auce', 'brbd', 'cato', 'chzh', 'cme', 'cnbe', 'defr']
Is 4 July 2025 a business day? False
Adjusted with Preceding: 2025-07-03


## Day-count conventions and contexts

In [3]:
from finstack.core.dates.schedule import Frequency, ScheduleBuilder, StubKind
from finstack.core.dates.daycount import DayCount, DayCountContext, DayCountContextState
act_365 = DayCount.from_name("ACT_365F")
quarterly = Frequency.from_months(3)
ctx = DayCountContext(calendar=us_calendar, frequency=quarterly)

accrual = act_365.year_fraction(date(2025, 1, 15), date(2025, 4, 15), ctx)
print("ACT/365 accrual (quarter):", round(accrual, 6))


ACT/365 accrual (quarter): 0.246575


## Persisting contexts via `DayCountContextState`

In [4]:
from finstack.core.dates.daycount import DayCount, DayCountContext, DayCountContextState
ctx_state = ctx.to_state()
json_payload = ctx_state.to_json()
restored_state = DayCountContextState.from_json(json_payload)
restored_ctx = restored_state.to_context()

print("Serialized context:", json_payload)
print("Round-trip frequency months:", restored_ctx.frequency.months)


Serialized context: {
  "calendar_id": "usny",
  "frequency": {
    "Months": 3
  },
  "bus_basis": null
}
Round-trip frequency months: 3


## Frequencies and stub handling

In [5]:
semiannual = Frequency.from_months(6)
short_first = StubKind.SHORT_FRONT
no_stub = StubKind.NONE
print("Semiannual months:", semiannual.months)
print("Stub options:", short_first.name, no_stub.name)


Semiannual months: 6
Stub options: short_front none


## ScheduleBuilder and period plans

In [6]:
from finstack.core.dates.periods import PeriodId
builder = ScheduleBuilder.new(date(2025, 1, 15), date(2026, 7, 15))
coupon_schedule = (
    builder
    .frequency(semiannual)
    .stub_rule(short_first)
    .adjust_with(dates.BusinessDayConvention.MODIFIED_FOLLOWING, us_calendar)
    .end_of_month(False)
    .build()
)
print("Schedule dates:")
for dt in coupon_schedule.dates:
    print(" ·", dt)

quarter_ids = [PeriodId.quarter(2025, q) for q in range(1, 5)]
print("Quarterly period IDs:", [pid.code for pid in quarter_ids])


Schedule dates:
 · 2025-01-15
 · 2025-07-15
 · 2026-01-15
 · 2026-07-15
Quarterly period IDs: ['2025Q1', '2025Q2', '2025Q3', '2025Q4']


## Date utilities

In [7]:
from finstack.core.dates.utils import date_to_days_since_epoch, days_since_epoch_to_date

epoch_days = date_to_days_since_epoch(date(2025, 1, 1))
restored_date = days_since_epoch_to_date(epoch_days)
print("Epoch days:", epoch_days, "->", restored_date)


Epoch days: 20089 -> 2025-01-01



## Checklist · best practices

- Always pair calendars with explicit business-day conventions when rolling schedules
- Persist `DayCountContextState` / schedule parameters along with market data so accrual logic can be reconstructed
- Favor `ScheduleBuilder` and `periods.build_periods` over handwritten date lists
- Use epoch-day utilities when storing dates in columnar stores or hashes
